# Task 1: Egg segmentation with a pretrained model (Grounding DINO + SAM)

The Roboflow export has **no box or mask labels**, so SAM can't be prompted from annotations.
The pipeline is:

1. **Grounding DINO** (open-vocabulary detector) with the text prompt `"an egg."` gives one box per egg.
2. The boxes are cleaned up: NMS, dropping tiny boxes, and dropping "group" boxes that enclose several eggs.
3. **SAM ViT-H** is prompted with each box and returns one mask per egg.
4. Each mask is post-processed (largest connected component, holes filled). Where masks overlap, a pixel goes to the mask SAM is more confident in.
5. Images the detector gets wrong are fixed with hand-written box or point prompts (`MANUAL_PROMPTS`) after reviewing the sheet.

**Output format**: `sam_masks/<image>.png` is an *instance label map* (uint8; 0 = background, 1..N = egg id).
The binary mask is `label > 0`. `sam_meta/<image>.json` stores boxes and scores, and `sam_overlays/` holds visual checks.

### Before running on Kaggle
* **Settings → Accelerator → GPU T4 x2** (or a single T4). Avoid P100: recent PyTorch builds dropped support for it.
* **Settings → Internet → On**.
* **Add-ons → Secrets** → add `ROBOFLOW_API_KEY` and attach it to this notebook.

In [ ]:
!pip install -q roboflow "git+https://github.com/facebookresearch/segment-anything.git"

## Configuration

In [ ]:
# [config]
import os, re, json, shutil, inspect
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()
DATA_DIR = WORK / 'data'
IMG_DIR = DATA_DIR / 'images'
OUT = WORK / 'sam_outputs'
WEIGHTS = WORK / 'weights'

# Detector
GD_MODEL_ID = 'IDEA-Research/grounding-dino-base'
TEXT_PROMPT = 'an egg.'        # Grounding DINO expects lowercase phrases ending with '.'
BOX_THRESHOLD = 0.35           # min box confidence
TEXT_THRESHOLD = 0.25          # min token-match confidence
NMS_IOU = 0.5                  # suppress duplicate boxes on the same egg
MIN_BOX_AREA_FRAC = 0.002      # drop boxes smaller than 0.2% of the image (specks, reflections)
CONTAIN_THR = 0.9              # box j is "inside" box i if >=90% of j lies within i

# Mask cleanup
MIN_COMPONENT_FRAC = 0.15      # keep mask pieces >= 15% of the largest piece (an occluder such as
                               # a rod or a leaf can split one egg into several pieces)

# Segmenter
SAM_TYPE = 'vit_h'
SAM_URLS = {
    'vit_h': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
    'vit_l': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_l_0b3195.pth',
    'vit_b': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
}

## Download the dataset from Roboflow
The API key is read from Kaggle Secrets (or the `ROBOFLOW_API_KEY` environment variable when running locally).

In [ ]:
# [download]
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
except Exception:
    api_key = os.environ['ROBOFLOW_API_KEY']

from roboflow import Roboflow
rf = Roboflow(api_key=api_key)
project = rf.workspace('fire-bf7qe').project('egg-vzhtr')
project.version(1).download('folder', location=str(DATA_DIR / 'raw'), overwrite=True)

# Flatten train/valid/test class folders: not_damaged_101_jpg.rf.<hash>.jpg -> not_damaged_101.jpg
IMG_DIR.mkdir(parents=True, exist_ok=True)
for f in sorted((DATA_DIR / 'raw').rglob('*.jpg')):
    shutil.copy2(f, IMG_DIR / (re.sub(r'_jpg\.rf\..*$', '', f.stem) + '.jpg'))

In [ ]:
def frame_no(p):
    return int(re.search(r'(\d+)$', Path(p).stem).group(1))

paths = sorted(IMG_DIR.glob('*.jpg'), key=frame_no)
print(len(paths), 'images')

## Load models

In [ ]:
import numpy as np, cv2, torch
from PIL import Image
from torchvision.ops import nms
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from segment_anything import sam_model_registry, SamPredictor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')

gd_proc = AutoProcessor.from_pretrained(GD_MODEL_ID)
gd_model = AutoModelForZeroShotObjectDetection.from_pretrained(GD_MODEL_ID).to(device).eval()

WEIGHTS.mkdir(parents=True, exist_ok=True)
sam_ckpt = WEIGHTS / Path(SAM_URLS[SAM_TYPE]).name
if not sam_ckpt.exists():
    torch.hub.download_url_to_file(SAM_URLS[SAM_TYPE], str(sam_ckpt))
sam = sam_model_registry[SAM_TYPE](checkpoint=str(sam_ckpt)).to(device).eval()
predictor = SamPredictor(sam)

## Pipeline functions

In [ ]:
# transformers renamed box_threshold -> threshold; support both versions.
_pp_params = inspect.signature(gd_proc.post_process_grounded_object_detection).parameters
_THR_KW = 'threshold' if 'threshold' in _pp_params else 'box_threshold'


def drop_group_boxes(boxes, scores):
    """Remove boxes that enclose >= 2 other boxes (Grounding DINO often adds one box around a group of eggs)."""
    n = len(boxes)
    if n < 3:
        return boxes, scores
    x1 = torch.maximum(boxes[:, None, 0], boxes[None, :, 0])
    y1 = torch.maximum(boxes[:, None, 1], boxes[None, :, 1])
    x2 = torch.minimum(boxes[:, None, 2], boxes[None, :, 2])
    y2 = torch.minimum(boxes[:, None, 3], boxes[None, :, 3])
    inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)
    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    inside = inter / area[None, :] >= CONTAIN_THR   # inside[i, j]: box j lies inside box i
    inside.fill_diagonal_(False)
    keep = inside.sum(1) < 2
    return boxes[keep], scores[keep]


@torch.no_grad()
def detect_eggs(img_rgb):
    h, w = img_rgb.shape[:2]
    inputs = gd_proc(images=Image.fromarray(img_rgb), text=TEXT_PROMPT, return_tensors='pt').to(device)
    out = gd_model(**inputs)
    r = gd_proc.post_process_grounded_object_detection(
        out, inputs.input_ids, text_threshold=TEXT_THRESHOLD, target_sizes=[(h, w)],
        **{_THR_KW: BOX_THRESHOLD})[0]
    boxes, scores = r['boxes'].float().cpu(), r['scores'].float().cpu()
    if len(boxes):
        keep = nms(boxes, scores, NMS_IOU)
        boxes, scores = boxes[keep], scores[keep]
        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        m = area >= MIN_BOX_AREA_FRAC * h * w
        boxes, scores = boxes[m], scores[m]
        boxes, scores = drop_group_boxes(boxes, scores)
    return boxes.numpy(), scores.numpy()


def clean_mask(m):
    """Drop small stray pieces and fill holes (eggs are solid blobs).

    Pieces smaller than MIN_COMPONENT_FRAC of the largest one are removed. Large pieces are all
    kept, because an occluder (a rod in #117, leaves in #10-13) splits one egg into separate parts.
    """
    m = m.astype(np.uint8)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n <= 1:
        return np.zeros(m.shape, bool)
    areas = stats[1:, cv2.CC_STAT_AREA]
    keep = 1 + np.flatnonzero(areas >= MIN_COMPONENT_FRAC * areas.max())
    m = np.isin(lab, keep).astype(np.uint8)
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    filled = np.zeros_like(m)
    cv2.drawContours(filled, cnts, -1, 1, thickness=cv2.FILLED)
    return filled.astype(bool)


@torch.no_grad()
def segment(img_rgb, prompts):
    """Prompted SAM -> instance label map. Overlapping pixels go to the mask with the higher predicted IoU.

    prompts: list of dicts {'box': [x1,y1,x2,y2], 'score': float, 'source': str,
                            optional 'pos': [[x,y],...], 'neg': [[x,y],...]}
    """
    h, w = img_rgb.shape[:2]
    label = np.zeros((h, w), np.uint8)
    best = np.full((h, w), -1.0, np.float32)
    insts = []
    if not prompts:
        return label, insts
    predictor.set_image(img_rgb)
    for pr in prompts:
        pts = [*pr.get('pos', []), *pr.get('neg', [])]
        lbl = [1] * len(pr.get('pos', [])) + [0] * len(pr.get('neg', []))
        masks, iou_pred, _ = predictor.predict(
            point_coords=np.array(pts, np.float32) if pts else None,
            point_labels=np.array(lbl) if pts else None,
            box=np.array(pr['box'], np.float32), multimask_output=False)
        m, s = clean_mask(masks[0]), float(iou_pred[0])
        if not m.any():
            continue
        idx = len(insts) + 1
        take = m & (s > best)
        label[take] = idx
        best[take] = s
        insts.append({'id': idx, 'box': [round(float(v), 1) for v in pr['box']], 'source': pr['source'],
                      'det_score': round(float(pr['score']), 3), 'sam_iou_pred': round(s, 3)})
    for inst in insts:   # final areas after overlap resolution
        inst['area'] = int((label == inst['id']).sum())
    return label, insts


COLORS = [(255, 64, 64), (64, 200, 64), (64, 128, 255), (255, 200, 0), (200, 64, 255), (0, 220, 220)]


def overlay(img_rgb, label, insts):
    vis = img_rgb.copy()
    for inst in insts:
        c = COLORS[(inst['id'] - 1) % len(COLORS)]
        m = label == inst['id']
        vis[m] = (0.55 * vis[m] + 0.45 * np.array(c)).astype(np.uint8)
        cnts, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(vis, cnts, -1, c, 2)
        x1, y1, x2, y2 = map(int, inst['box'])
        cv2.rectangle(vis, (x1, y1), (x2, y2), (255, 255, 0), 1)
        cv2.putText(vis, f"{inst['id']}:{inst['det_score']:.2f}", (x1 + 2, max(14, y1 + 14)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 0), 1, cv2.LINE_AA)
    return vis

## Manual prompt fixes (from reviewing the first run)
After the first full run, three images had problems that a threshold change couldn't fix without adding false
boxes everywhere else. Each fix is a prompt for SAM (pixel coordinates on the 640×640 image):

* `'add'`: extra eggs added on top of the detector's boxes.
* `'replace'`: the detector's boxes are thrown away and these prompts are used instead.
* Each prompt has a `box` and optional `pos` / `neg` points (clicks for "egg here" / "not egg here").

Every fix is recorded as `source: manual` in the meta JSON and listed in the report.

In [ ]:
MANUAL_PROMPTS = {
    # Grounding DINO missed the bottom-left egg (probably scored below BOX_THRESHOLD; it is in shadow).
    'not_damaged_105': {'add': [{'box': [20, 360, 255, 505]}]},
    'not_damaged_134': {'add': [{'box': [0, 335, 295, 515]}]},
    # One egg in a tray cup, split by the grey rod. With the box alone SAM picked the cup rim, so add a
    # positive click on each side of the rod.
    'not_damaged_117': {'replace': [{'box': [245, 105, 525, 295], 'pos': [[300, 195], [470, 185]]}]},
}


def build_prompts(stem, img_rgb):
    fix = MANUAL_PROMPTS.get(stem, {})
    prompts = []
    if 'replace' not in fix:
        boxes, scores = detect_eggs(img_rgb)
        prompts = [{'box': b.tolist(), 'score': float(s), 'source': 'grounding_dino'} for b, s in zip(boxes, scores)]
    for pr in fix.get('replace', []) + fix.get('add', []):
        prompts.append({**pr, 'score': 1.0, 'source': 'manual'})
    return prompts

## Run on all images

In [ ]:
from tqdm.auto import tqdm

for sub in ['sam_masks', 'sam_meta', 'sam_overlays']:
    (OUT / sub).mkdir(parents=True, exist_ok=True)

summary = []
for p in tqdm(paths):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    label, insts = segment(img, build_prompts(p.stem, img))

    cv2.imwrite(str(OUT / 'sam_masks' / f'{p.stem}.png'), label)
    cv2.imwrite(str(OUT / 'sam_overlays' / f'{p.stem}.jpg'), cv2.cvtColor(overlay(img, label, insts), cv2.COLOR_RGB2BGR))
    with open(OUT / 'sam_meta' / f'{p.stem}.json', 'w') as f:
        json.dump({'image': p.name, 'manual_fix': MANUAL_PROMPTS.get(p.stem), 'instances': insts}, f, indent=1)
    summary.append({'image': p.stem, 'n_eggs': len(insts),
                    'n_manual': sum(i['source'] == 'manual' for i in insts),
                    'min_det': min([i['det_score'] for i in insts], default=None),
                    'fg_frac': round(float((label > 0).mean()), 4)})

import pandas as pd
df = pd.DataFrame(summary)
df.to_csv(OUT / 'summary.csv', index=False)
print(df['n_eggs'].value_counts().sort_index())
print('No eggs found in:', df.loc[df.n_eggs == 0, 'image'].tolist())

## Review sheet
Check every tile. Faint boxes or low scores, missing eggs and masks that leak into the background are what `MANUAL_PROMPTS` is for.

In [ ]:
import matplotlib.pyplot as plt

def review_sheet(stems, cols=8, tile=240):
    tiles = []
    for s in stems:
        im = cv2.resize(cv2.imread(str(OUT / 'sam_overlays' / f'{s}.jpg')), (tile, tile))
        cv2.putText(im, s.replace('not_damaged_', '#'), (4, tile - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
        tiles.append(im)
    while len(tiles) % cols:
        tiles.append(np.zeros_like(tiles[0]))
    return np.vstack([np.hstack(tiles[i:i + cols]) for i in range(0, len(tiles), cols)])

stems = df['image'].tolist()
for start in range(0, len(stems), 40):
    sheet = review_sheet(stems[start:start + 40])
    cv2.imwrite(str(OUT / f'review_sheet_{start // 40}.jpg'), sheet)
    plt.figure(figsize=(20, 20 * sheet.shape[0] / sheet.shape[1]))
    plt.imshow(cv2.cvtColor(sheet, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()

## Package outputs
Download `sam_outputs.zip` from the Output panel and unzip it into `outputs/` in the repo (giving `outputs/sam_masks/` and so on).

In [ ]:
with open(OUT / 'params.json', 'w') as f:
    json.dump({'gd_model': GD_MODEL_ID, 'text_prompt': TEXT_PROMPT, 'box_threshold': BOX_THRESHOLD,
               'text_threshold': TEXT_THRESHOLD, 'nms_iou': NMS_IOU, 'min_box_area_frac': MIN_BOX_AREA_FRAC,
               'contain_thr': CONTAIN_THR, 'min_component_frac': MIN_COMPONENT_FRAC, 'sam_type': SAM_TYPE,
               'manual_prompts': MANUAL_PROMPTS}, f, indent=1)
shutil.make_archive(str(WORK / 'sam_outputs'), 'zip', OUT)
print('wrote', WORK / 'sam_outputs.zip')